In [3]:
import pandas as pd  # Import pandas for data handling
import numpy as np  # Import numpy for numerical operations
import joblib  # Import joblib for loading the model and scaler
from urllib.parse import urlparse  # Import URL parser
import re  # Import regular expressions


model = joblib.load(
    "../data/models/final_model.pkl"
)  # Load the trained XGBoost model

scaler = joblib.load(
    "../data/processed/scaler.pkl"
)  # Load the fitted scaler


def extract_features(url):  # Create function to extract the 22 required features

    parsed_url = urlparse(url)  # Break URL into its components

    domain = parsed_url.netloc  # Extract domain from URL

    path = parsed_url.path  # Extract path from URL

    query = parsed_url.query  # Extract query string from URL

    url_length = len(url)  # Calculate total URL length

    domain_length = len(domain)  # Calculate domain length

    is_ip = int(bool(re.fullmatch(
        r"(?:\d{1,3}\.){3}\d{1,3}",
        domain.split(":")[0]
    )))  # Check whether domain is an IP address

    parts = domain.split(".")  # Split domain into components

    tld = ".".join(parts[-2:]) if len(parts) >= 2 else ""  # Extract TLD

    tld_length = len(tld)  # Calculate TLD length

    subdomain_count = max(len(parts) - 2, 0)  # Count subdomains

    letter_count = sum(c.isalpha() for c in url)  # Count letters

    digit_count = sum(c.isdigit() for c in url)  # Count digits

    special_count = sum(
        not c.isalnum() for c in url
    )  # Count special characters

    eq_count = url.count("=")  # Count equal signs

    qm_count = url.count("?")  # Count question marks

    amp_count = url.count("&")  # Count ampersands

    dot_count = url.count(".")  # Count dots

    dash_count = url.count("-")  # Count hyphens

    under_count = url.count("_")  # Count underscores

    letter_ratio = letter_count / url_length if url_length else 0  # Calculate letter ratio

    digit_ratio = digit_count / url_length if url_length else 0  # Calculate digit ratio

    special_ratio = special_count / url_length if url_length else 0  # Calculate special character ratio

    is_https = int(parsed_url.scheme.lower() == "https")  # Check HTTPS usage

    slash_count = url.count("/")  # Count slashes

    character_counts = pd.Series(list(url)).value_counts()  # Count URL characters

    probabilities = character_counts / len(url) if url else pd.Series(dtype=float)  # Calculate character probabilities

    entropy = -sum(
        probabilities * np.log2(probabilities)
    ) if url else 0  # Calculate URL entropy

    path_length = len(path)  # Calculate path length

    query_length = len(query)  # Calculate query length


    features = [
        url_length,
        domain_length,
        is_ip,
        tld_length,
        subdomain_count,
        letter_count,
        digit_count,
        special_count,
        eq_count,
        qm_count,
        amp_count,
        dot_count,
        dash_count,
        under_count,
        letter_ratio,
        digit_ratio,
        special_ratio,
        is_https,
        slash_count,
        entropy,
        path_length,
        query_length
    ]  # Store the 22 extracted features

    return np.array(features).reshape(1, -1)  # Return features in model input format


url = "https://example.com"  # Test URL

features = extract_features(url)  # Extract features from test URL

features_scaled = scaler.transform(features)  # Scale features using the trained scaler

prediction = model.predict(features_scaled)[0]  # Generate model prediction

probability = model.predict_proba(features_scaled)[0]  # Generate prediction probabilities

print("URL:", url)  # Display tested URL

print("Prediction:", "Phishing" if prediction == 1 else "Legitimate")  # Display prediction

print("Phishing Probability:", probability[1])  # Display phishing probability

URL: https://example.com
Prediction: Legitimate
Phishing Probability: 0.0003657181


/Users/pranavanand/Desktop/Adaptive-Phishing-Threat-Intelligence-System/venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
